# Avaliação de transcrições
Compare respostas LLM e OCR com um gabarito. Use links públicos do Drive.


In [11]:
%pip -q install "jiwer>=4,<5" pandas plotly "gdown>=6,<7"

Note: you may need to restart the kernel to use updated packages.


## 1. Configuração
Informe o link do gabarito e das duas pastas de respostas. Todos devem ser públicos.


In [12]:
import json
import re
import unicodedata
from pathlib import Path

import gdown
import jiwer
import pandas as pd
import plotly.express as px
from IPython.display import display

LINK_GABARITOS = "https://drive.google.com/file/d/14VlyCmLAeg_zAHeh6wXz6SPjZ7cHDcDS/view?usp=sharing"
LINK_PASTA_LLM = "https://drive.google.com/drive/folders/1SjAmynei_WxJKISSxJ_9larKlNEcmjEq?usp=sharing"
LINK_PASTA_OCR_PIPELINE = "https://drive.google.com/drive/folders/1IJWp6P-316jIryIMVturGFRMG1gFd2QE?usp=sharing"
METRICA_ORDENACAO = "cer_sem_pontuacao"


## 2. Normalização e métricas
As funções de tratamento dos textos e o cálculo de PER permanecem iguais aos do avaliador anterior.


In [13]:
_PUNCT_CHAR = r"(?!(?<=\w)['’-](?=\w))[^\w\s]"
_PUNCT_SUB = re.compile(f"({_PUNCT_CHAR})")
_IS_PUNCT = re.compile(r"^[^\w\s]$")
PUNCT = "[PUNCT]"


def normalize_text(text):
    text = text.lower()
    text = re.sub(_PUNCT_CHAR, "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_separate_punctuation(text):
    text = text.lower()
    text = _PUNCT_SUB.sub(r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


In [14]:
def _mask(tokens):
  return [PUNCT if _IS_PUNCT.match(t) else t for t in tokens]


def _align_punctuation(ref, hyp):
  """Alinha as sequências mascaradas e devolve os pares (i, j) de
  pontuações alinhadas entre si."""
  ref_m, hyp_m = _mask(ref), _mask(hyp)
  n, m = len(ref), len(hyp)
  big = n + m + 1

  def step(i, j):
    if ref_m[i] != hyp_m[j]:
      return big
    if ref_m[i] == PUNCT and ref[i] != hyp[j]:
      return 1
    return 0

  dist = [[0] * (m + 1) for _ in range(n + 1)]
  for i in range(1, n + 1):
    dist[i][0] = i * big
  for j in range(1, m + 1):
    dist[0][j] = j * big
  for i in range(1, n + 1):
    for j in range(1, m + 1):
        dist[i][j] = min(
          dist[i - 1][j - 1] + step(i - 1, j - 1),
          dist[i - 1][j] + big,
          dist[i][j - 1] + big,
        )

  pairs, i, j = [], n, m
  while i > 0 and j > 0:
    if dist[i][j] == dist[i - 1][j - 1] + step(i - 1, j - 1):
      if ref_m[i - 1] == PUNCT and hyp_m[j - 1] == PUNCT:
          pairs.append((i - 1, j - 1))
      i, j = i - 1, j - 1
    elif dist[i][j] == dist[i - 1][j] + big:
      i -= 1
    else:
      j -= 1
  return pairs


def punctuation_counts(reference, hypothesis):
  """Espera textos já passados por normalize_separate_punctuation."""
  ref, hyp = reference.split(), hypothesis.split()
  pairs = _align_punctuation(ref, hyp)

  n_ref = sum(1 for t in ref if _IS_PUNCT.match(t))
  n_hyp = sum(1 for t in hyp if _IS_PUNCT.match(t))
  correct = sum(ref[i] == hyp[j] for i, j in pairs)
  return {
    "C": correct,
    "S": len(pairs) - correct,
    "D": n_ref - len(pairs),
    "I": n_hyp - len(pairs),
  }


def _per(c):
  errors = c["S"] + c["D"] + c["I"]
  total = errors + c["C"]
  return errors / total if total else 0.0


def punctuation_error_rate(reference, hypothesis):
  """
  Cálculo da Punctuation Error Rate (PER), com base em https://arxiv.org/abs/2310.02943

  PER = Ip + Dp + Sp / Ip + Dp + Sp + Cp
  Ip = Pontuações Inseridas
  Dp = Pontuações Deletadas
  Sp = Pontuações Substituídas
  Cp = Pontuações Corretas
  """
  reference = normalize_separate_punctuation(reference)
  hypothesis = normalize_separate_punctuation(hypothesis)
  return _per(punctuation_counts(reference, hypothesis))


def corpus_punctuation_error_rate(references, hypotheses):
  """ PER de um corpus todo"""
  total = {"C": 0, "S": 0, "D": 0, "I": 0}
  for r, h in zip(references, hypotheses, strict=True):
    r = normalize_separate_punctuation(r)
    h = normalize_separate_punctuation(h)
    for k, v in punctuation_counts(r, h).items():
      total[k] += v
  return _per(total)


## 3. Carregar os JSONs


In [ ]:
def normalize_filename(filename):
    return unicodedata.normalize("NFC", filename.replace("\\", "/").rsplit("/", 1)[-1])


pasta_entradas = Path("entradas").resolve()
pasta_entradas.mkdir(exist_ok=True)
ARQUIVO_GABARITOS = pasta_entradas / "gabarito.json"
gdown.download(LINK_GABARITOS, output=str(ARQUIVO_GABARITOS), use_cookies=False)
dados = json.loads(ARQUIVO_GABARITOS.read_text(encoding="utf-8-sig"))
gabaritos_por_arquivo = {normalize_filename(nome): texto for nome, texto in dados.items()}

fontes_respostas = []
for grupo, link in (("LLM", LINK_PASTA_LLM), ("OCR Pipeline", LINK_PASTA_OCR_PIPELINE)):
    caminhos = gdown.download_folder(link, output=str(pasta_entradas / grupo), use_cookies=False)
    for caminho in sorted(map(Path, caminhos)):
        if caminho.suffix.lower() != ".json":
            continue
        id_resposta = f"r{len(fontes_respostas) + 1:02d}"
        fontes_respostas.append({
            "id_resposta": id_resposta, "arquivo_respostas": caminho.name,
            "link_drive": link, "grupo": grupo, "caminho": caminho,
        })
        print(f"{id_resposta}: {caminho.name}")


## 4. Avaliar


In [ ]:
METRICAS = [
    "wer_sem_pontuacao", "cer_sem_pontuacao",
    "wer_com_pontuacao", "cer_com_pontuacao", "per",
]
COLUNAS_CONTEXTO = ["id_resposta", "arquivo_respostas", "link_drive", "grupo", "modelo"]


def calcular_metricas(referencias, hipoteses):
    if not referencias:
        return dict.fromkeys(METRICAS)
    refs_sem = [normalize_text(t) for t in referencias]
    hyps_sem = [normalize_text(t) for t in hipoteses]
    refs_com = [normalize_separate_punctuation(t) for t in referencias]
    hyps_com = [normalize_separate_punctuation(t) for t in hipoteses]
    return {
        "wer_sem_pontuacao": jiwer.wer(refs_sem, hyps_sem),
        "cer_sem_pontuacao": jiwer.cer(refs_sem, hyps_sem),
        "wer_com_pontuacao": jiwer.wer(refs_com, hyps_com),
        "cer_com_pontuacao": jiwer.cer(refs_com, hyps_com),
        "per": corpus_punctuation_error_rate(referencias, hipoteses),
    }


def avaliar_resposta(fonte, gabaritos):
    resultados = json.loads(fonte["caminho"].read_text(encoding="utf-8-sig"))
    contexto = {k: fonte[k] for k in COLUNAS_CONTEXTO if k != "modelo"}
    contexto["modelo"] = resultados["modelo"]
    linhas, referencias, hipoteses = [], [], []
    for imagem in resultados["imagens"]:
        nome = normalize_filename(imagem["arquivo"])
        referencia, transcricao = gabaritos[nome], imagem["texto_transcrito"]
        linhas.append({**contexto, "arquivo": nome, **calcular_metricas([referencia], [transcricao])})
        referencias.append(referencia)
        hipoteses.append(transcricao)
    resumo = {**contexto, **calcular_metricas(referencias, hipoteses)}
    return {"linhas": linhas, "resumo": resumo}


avaliacoes = [avaliar_resposta(fonte, gabaritos_por_arquivo) for fonte in fontes_respostas]
todas_linhas = [linha for avaliacao in avaliacoes for linha in avaliacao["linhas"]]
resumos = [avaliacao["resumo"] for avaliacao in avaliacoes]


## 5. Comparar


In [ ]:
nomes_imagens = [linha["arquivo"] for linha in avaliacoes[0]["linhas"]]
df_resumo = pd.DataFrame(resumos)
df_imagens = pd.DataFrame(todas_linhas)
df_ranking = df_resumo.sort_values([METRICA_ORDENACAO, "id_resposta"]).copy()
rotulos = {r["id_resposta"]: f"{r['id_resposta']} - {r['modelo'].rsplit('/', 1)[-1]}" for r in resumos}
df_ranking["rotulo"] = df_ranking["id_resposta"].map(rotulos)


## 6. Ranking
Erros em porcentagem. Valores menores indicam melhor desempenho.


In [ ]:
NOMES_METRICAS = {
    "wer_sem_pontuacao": "WER s/ pont.", "cer_sem_pontuacao": "CER s/ pont.",
    "wer_com_pontuacao": "WER c/ pont.", "cer_com_pontuacao": "CER c/ pont.", "per": "PER",
}
NOMES_COLUNAS = {
    "id_resposta": "ID", "grupo": "Grupo", "modelo": "Modelo", "arquivo_respostas": "Arquivo JSON",
    **NOMES_METRICAS,
}
CORES_GRUPOS = {"LLM": "#2563eb", "OCR Pipeline": "#ea580c"}
ALTURA_GRAFICOS = max(360, 55 * len(avaliacoes) + 160)


def mostrar_tabela(df, colunas):
    tabela = df[colunas].copy()
    for coluna in METRICAS:
        if coluna in tabela:
            tabela[coluna] = tabela[coluna].map(lambda v: "—" if pd.isna(v) else f"{v:.2%}")
    tabela = tabela.rename(columns=NOMES_COLUNAS).fillna("—").set_index("ID")
    with pd.option_context("display.max_columns", None, "display.max_colwidth", 45):
        display(tabela)


def mostrar_grafico(figura, titulo):
    figura.update_layout(
        title=titulo, template="plotly_white", height=ALTURA_GRAFICOS,
        margin=dict(l=20, r=20, t=70, b=60), font=dict(size=12),
        legend=dict(orientation="h", y=-0.2, x=0),
    )
    figura.update_yaxes(autorange="reversed", automargin=True)
    figura.show(config={"displaylogo": False, "responsive": True})


mostrar_tabela(df_ranking, ["id_resposta", "grupo", "modelo", *METRICAS])


In [ ]:
dados = df_ranking.assign(taxa_percentual=df_ranking[METRICA_ORDENACAO] * 100)
fig_ranking = px.bar(
    dados, x="taxa_percentual", y="rotulo", color="grupo", orientation="h",
    color_discrete_map=CORES_GRUPOS, text="taxa_percentual",
    hover_data=["modelo", "arquivo_respostas"],
    labels={"taxa_percentual": f"{NOMES_METRICAS[METRICA_ORDENACAO]} (%)", "rotulo": "Execução", "grupo": "Grupo"},
)
fig_ranking.update_traces(texttemplate="%{x:.2f}%", textposition="outside", cliponaxis=False)
fig_ranking.update_xaxes(range=[0, max(1, dados["taxa_percentual"].max() * 1.2)])
fig_ranking.update_yaxes(categoryorder="array", categoryarray=dados["rotulo"].tolist())
mostrar_grafico(fig_ranking, "Ranking")

fig_metricas = px.imshow(
    df_ranking[METRICAS].to_numpy(dtype=float) * 100,
    x=[NOMES_METRICAS[m] for m in METRICAS], y=df_ranking["rotulo"].tolist(),
    color_continuous_scale="YlOrRd", zmin=0, aspect="auto", text_auto=".2f",
    labels={"x": "Métrica", "y": "Execução", "color": "Erro (%)"},
)
fig_metricas.update_traces(hovertemplate="%{y}<br>%{x}: %{z:.2f}%<extra></extra>")
mostrar_grafico(fig_metricas, "Métricas (%)")


## 7. Respostas


In [ ]:
mostrar_tabela(df_resumo, [
    "id_resposta", "grupo", "arquivo_respostas", "modelo",
])


## 8. Resultados por imagem


In [ ]:
matriz = df_imagens.pivot(index="id_resposta", columns="arquivo", values="cer_sem_pontuacao")
matriz = matriz.reindex(index=df_ranking["id_resposta"], columns=nomes_imagens)
fig_imagens = px.imshow(
    matriz.to_numpy(dtype=float) * 100,
    x=[f"Imagem {i}" for i in range(1, len(nomes_imagens) + 1)],
    y=[rotulos[id_resposta] for id_resposta in matriz.index],
    color_continuous_scale="YlOrRd", zmin=0, aspect="auto",
    labels={"x": "Documento", "y": "Execução", "color": "CER (%)"},
)
fig_imagens.update_traces(
    customdata=[nomes_imagens for _ in matriz.index],
    hovertemplate="%{y}<br>%{customdata}<br>CER: %{z:.2f}%<extra></extra>",
)
mostrar_grafico(fig_imagens, "CER sem pontuação por imagem")
